In [ ]:
import ast
from datasets import Dataset
import os
import pandas as pd

In [ ]:
data_path = "checkpoints\partial_results.parquet"  

In [ ]:
if not os.path.exists(data_path):
    raise FileNotFoundError(f"❌ Файл не найден: {data_path}")

print(f"🔹 Загружаем датасет из {data_path}")
df = pd.read_parquet(data_path)
print(f"📦 Загружено {len(df)} строк, колонки: {list(df.columns)}")

# Проверка и фильтрация
required_cols = {"code", "docstring"}
if not required_cols.issubset(df.columns):
    raise ValueError(
        f"❌ В датасете должны быть колонки {required_cols}, найдено: {set(df.columns)}"
    )

# Оставляем только нужные колонки
df = df[list(required_cols)]

In [ ]:
df.drop_duplicates

In [ ]:
df.info

In [ ]:
df.to_parquet('git_clen_data.parquet')

In [ ]:
def remove_docstring(source_code):
    try:
        # Парсим код в AST
        tree = ast.parse(source_code)
        if isinstance(tree.body[0], ast.FunctionDef):
            # Убираем docstring, если он есть
            tree.body[0].body = [
                node for node in tree.body[0].body
                if not (isinstance(node, ast.Expr) and isinstance(node.value, ast.Str))
            ]
        # Возвращаем "очищенный" код
        return ast.unparse(tree)
    except Exception:
        return source_code  # если парсинг не удался, возвращаем исходное

In [ ]:
dataset = Dataset.from_pandas(df)

In [ ]:
dataset = dataset.map(lambda x: {"code": remove_docstring(x["code"])})

In [ ]:
dataset.to_parquet('git_dataset.parquet')